# Incremental capacity analysis (dQ/dV)
In this notebook we illustrate, how to use cellpy to extract dQ/dV data for selected cycles.

The respective methods are collected in the ica utilities (cellpy.utils.ica):

- **`ica.dqdv`**: This is the main and recommended method, taking data contained within the CellpyCell object as input.

- Additional methods allow for the calculation of dQ/dV data based on voltage and capacity data provided in different, cellpy-agnostic formats:
    - `ica.dqdv_cycle` and `ica.dqdv_cycles` takes Pandas DataFrames containing capacity vs voltage data for one or multiple cycles as input
    - `ica.dqdv_np` uses simple arrays or Pandas DataSeries of capacity and voltage as input.
    
For more details on these, have a look at the source code.

In [ ]:
import pathlib
from cellpy import prms

prms.Paths.examplesdir = pathlib.Path("./data/examples")

In [ ]:
import cellpy
from cellpy.utils import example_data, ica


import plotly.express as px


print(f"{cellpy.__version__ = }")

Load an example datafile:

In [ ]:
c = example_data.cellpy_file()

## Extracting dQ/dV data using `ica.dqdv`
This example shows how to get dQ/dV data directly and easily from the CellpyCell object (obtained by loading the data using `cellpy.get()`).

The dQ/dV data is provided as a Pandas DataFrame.

Without specifying any further options, the dQ/dV for all cycles contained within the CellpyCell is calculated:

In [ ]:
ica_df = ica.dqdv(c)
ica_df.head()

In [ ]:
px.line(
    ica_df,
    x="voltage",
    y="dq",
    color="cycle",
    range_x=[-0.01, 1.51],
    width=600
)

If cycle number(s) are specified using the `cycle` keyword (as an integer or list of integers), the dQ/dV will be calculated for those cycles only. If `split=True`, two separate Pandas Dataframes will be obtained, one containing charge, and one containing discharge data:

In [ ]:
cycles = [2, 3, 4]
charge_ica, discharge_ica = ica.dqdv(c, split=True, cycle=cycles)

In [ ]:
charge_ica.head(3)

In [ ]:
discharge_ica.head(3)

In [ ]:
px.line(charge_ica, x="voltage", y="dq", color="cycle", width=600)

### Tweaking the algorithm

In [ ]:
ica_df_1 = ica.dqdv(c, cycle=3, voltage_resolution=0.03)
ica_df_2 = ica.dqdv(c, cycle=3, voltage_resolution=0.01)
ica_df_3 = ica.dqdv(c, cycle=3, voltage_resolution=0.005)

In [ ]:
import pandas as pd

ica_both = pd.concat(
    [ica_df_1, ica_df_2, ica_df_3],
    keys=["0.03", "0.01", "0.005"],
    names=["v_res", "index"],
).reset_index()
px.line(
    ica_both,
    x="voltage",
    y="dq",
    color="v_res",
    range_x=[0.1, 0.4],
    range_y=[0, 6000],
    symbol="v_res",
    width=600
)

## Assignments

### A4.1

Try to reproduce something looking like this:

![title](static/example_ica.png)